# Bac a sable — pandas + SQL

- **ZONE SETUP** : les donnees, a lancer une fois (pandas: `commandes/clients/produits` ; SQL: `q("...")`).
- **ZONE VIDE** : tu ecris ici le code des exercices en cours (enonces en commentaire).
- **ARCHIVES** (en bas) : exos reussis, regroupes par THEME + DATE.

Aide-memoire : `fiches-revision/Carnet_de_recettes.md`.

## ZONE SETUP — executer une fois

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

commandes = pd.DataFrame({
    "client":   ["Alice","Alice","Bob","Bob","Chloe","Chloe","David","Alice","Bob","David"],
    "produit":  ["P1","P2","P1","P3","P2","P3","P1","P3","P2","P2"],
    "quantite": [2, 1, 5, 3, 2, 1, 4, np.nan, 2, 1],
    "prix_unitaire": [10.0, 30.0, 10.0, 100.0, 30.0, 100.0, 10.0, 100.0, 30.0, 30.0],
    "date":     ["2025-01-05","2025-01-07","2025-01-09","2025-02-01","2025-02-03",
                 "2025-02-10","2025-03-01","2025-03-05","2025-03-08","2025-03-20"],
})
commandes = pd.concat([commandes, commandes.iloc[[2]]], ignore_index=True)  # 1 doublon expres

clients = pd.DataFrame({
    "client":  ["Alice","Bob","Chloe","David"],
    "ville":   ["Paris","Lyon","Paris","Marseille"],
    "segment": ["Pro","Particulier","Pro","Particulier"],
})

produits = pd.DataFrame({
    "produit":   ["P1","P2","P3"],
    "categorie": ["Accessoire","Accessoire","Tech"],
    "prix_catalogue": [10.0, 30.0, 100.0],
})

conn = sqlite3.connect(":memory:")
commandes.to_sql("commandes", conn, index=False, if_exists="replace")
clients.to_sql("clients", conn, index=False, if_exists="replace")
produits.to_sql("produits", conn, index=False, if_exists="replace")

def q(sql):
    "Execute une requete SQL et renvoie un DataFrame."
    return pd.read_sql_query(sql, conn)

print("OK - commandes / clients / produits prets  |  SQL: q(\"SELECT ...\")")
commandes

## ZONE VIDE — ecris ton code ici

In [ ]:
# [SQL] Drill 1 — le client avec le 2e plus gros CA total.
# (indice : DENSE_RANK dans une CTE, puis WHERE rang = 2)



In [ ]:
# [SQL] Drill 2 — part (%) de chaque ville dans le CA TOTAL de toutes les villes.
# (indice : SUM par ville, et le total avec une fenetre SUM(...) OVER () ; ratio = part / total * 100)



In [ ]:
# [pandas] Drill 3 — CA total par MOIS (extraire le mois de la colonne date).
# (indice : pd.to_datetime(...) puis .dt.month ou .dt.to_period("M"))



In [ ]:
# [pandas] Drill 4 — nombre de CATEGORIES distinctes commandees par chaque client.
# (indice : merge produits -> groupby client -> nunique sur categorie)



---
# ARCHIVES — exos reussis, regroupes par THEME + DATE

**Convention :**
- 1 seul theme -> titre `THEME — DATE`, cellules dessous.
- Plusieurs themes -> titre `THEMES — DATE`, puis sous-titres `• Theme X — DATE`.
- Sous chaque theme : `A re-tester le ...` (date + 7 jours).

## pandas — regrouper & rapprocher (groupby / merge / agg) — 03/09/2026
**A re-tester le 10/09/2026**.

In [ ]:
# Exo : CA total par client, top 3.
commandes = commandes.dropna()                                    # (remarque: modifie la table de base + inutile ici)
print(commandes)
commandes["CA"] = commandes["quantite"] * commandes["prix_unitaire"]
commandes.groupby("client")["CA"].sum().nlargest(3)

In [ ]:
# Exo : CA total par categorie (merge commandes + produits).
commande_produit = commandes.merge(produits, on="produit")
commande_produit["CA"] = commande_produit["prix_unitaire"] * commande_produit["quantite"]
commande_produit.groupby("categorie")["CA"].sum()

In [ ]:
# Exo : par ville, nb de commandes + CA total.
commande_client = commandes.merge(clients, on="client")
commande_client["CA"] = commande_client["prix_unitaire"] * commande_client["quantite"]
commande_client = commande_client.groupby("ville").agg(
    nb_commande=("quantite", "count"),   # count ignore les NaN ; ("client","count") ou .size comptent TOUT
    CA_tot=("CA", "sum")
)
commande_client
# Amelioration : .sort_values("CA_tot", ascending=False)

## THEMES — 04/09/2026
**A re-tester le 11/09/2026**.

### • pandas — pivot_table / apply / panier moyen — 04/09/2026

In [ ]:
# pivot_table CA par ville x categorie.
commandes_client = commandes.merge(clients, on="client") \
                            .merge(produits, on="produit") \
                            .assign(CA=lambda x: x["prix_unitaire"]*x["quantite"])
commandes_client.pivot_table(index="ville", columns="categorie", values="CA", aggfunc="sum") \
                .fillna("Categorie indisponible dans cette ville")
# Plus propre : fill_value=0 DANS pivot_table.

In [ ]:
# colonne "gamme" = premium si prix_unitaire >= 100 sinon standard.
commandes["gamme"] = commandes.apply(lambda x: "premium" if x["prix_unitaire"] >= 100 else "standard", axis=1)
commandes
# Variante rapide : np.where(commandes["prix_unitaire"]>=100, "premium", "standard")

In [ ]:
# panier moyen (CA moyen par commande) par segment.
commandes_client_produit = commandes.assign(CA=lambda d: d.prix_unitaire * d.quantite) \
                                    .merge(clients, on="client") \
                                    .merge(produits, on="produit")
commandes_client_produit = commandes_client_produit.groupby("segment")["CA"].mean()
display(commandes_client_produit)

### • SQL — window & CTE (RANK, cumul, CTE) — 04/09/2026

In [ ]:
# RANK : classer les clients par CA total.
q("""
  SELECT client, SUM(quantite*prix_unitaire), RANK() over (order by SUM(quantite*prix_unitaire) desc)
  FROM commandes co
  GROUP BY client
  """)
# Amelioration : nommer les colonnes -> AS ca, AS rang.

In [ ]:
# cumul du CA par date (running total). Cle : SUM(SUM(...)) OVER.
q("""
  SELECT date,
         SUM(prix_unitaire*quantite) as CA,
         SUM(SUM(prix_unitaire * quantite)) over (ORDER BY date) as CA_cumulee
  FROM commandes
  GROUP BY date
  """)

In [ ]:
# CTE : clients dont le CA total depasse le CA moyen.
q("""
  WITH calcul_CA_moyen AS (
      SELECT co.client, SUM(co.prix_unitaire * co.quantite) as CA_tot
      FROM commandes co
      group by co.client              -- sans GROUP BY : SUM agrege tout en 1 ligne
  )
  SELECT *
  FROM calcul_CA_moyen
  WHERE CA_tot > (SELECT AVG(CA_tot) from calcul_CA_moyen)
  """)

## THEMES — 07/09/2026
**A re-tester le 14/09/2026**. (Note : Exo ROW_NUMBER en SQL et top-produit en pandas = le MEME motif "meilleur par groupe".)

### • SQL — HAVING & top-1 par groupe (ROW_NUMBER) — 07/09/2026

In [ ]:
# Exo : villes dont le CA total depasse 200 EUR (HAVING = filtrer les groupes).
q("""
  SELECT cl.ville, SUM(co.quantite * co.prix_unitaire) as CA_total
  FROM commandes co
  JOIN clients cl on co.client = cl.client
  GROUP BY cl.ville
  HAVING SUM(co.quantite * co.prix_unitaire) > 200
  """)

In [ ]:
# Exo : la commande la plus chere de chaque client (ROW_NUMBER = 1 par groupe).
q("""
  WITH rang_commande AS (
        SELECT client, (quantite*prix_unitaire) as prix_commande,
               ROW_NUMBER() OVER (PARTITION BY client ORDER BY (quantite*prix_unitaire) DESC) as rang
        FROM commandes
  )
  SELECT *
  FROM rang_commande
  WHERE rang = 1
  """)

### • pandas — top par groupe & json_normalize — 07/09/2026

In [ ]:
# Exo : top produit (par CA) dans chaque categorie.
# Motif = equivalent pandas de ROW_NUMBER()=1 : trier decroissant PUIS groupby(groupe).head(1).
commandes_produits = commandes.merge(produits, on="produit") \
                              .assign(CA=lambda x: x.quantite*x.prix_unitaire) \
                              .groupby(["categorie", "produit"], as_index=False)["CA"].sum() \
                              .sort_values("CA", ascending=False) \
                              .groupby("categorie").head(1)
commandes_produits

In [ ]:
# Exo : API -> pandas, aplatir un JSON imbrique avec json_normalize.
data = [
    {"id": 1, "client": {"nom": "Alice", "ville": "Paris"}, "total": 120},
    {"id": 2, "client": {"nom": "Bob",   "ville": "Lyon"},  "total": 80},
]
df_aplati = pd.json_normalize(data, sep="_")   # sous-objet "client" -> colonnes client_nom, client_ville
df_aplati